In [1]:
# ============================================================
# 1. IMPORT LIBRARIES AND CHECK GPU
# ============================================================

import os
import numpy as np
import pandas as pd
import torch

from pathlib import Path
from IPython.display import display

print("Libraries imported successfully.")

print("\nPyTorch version:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    print(
        "GPU memory:",
        round(
            torch.cuda.get_device_properties(0).total_memory / (1024**3),
            2
        ),
        "GB"
    )
else:
    raise RuntimeError(
        "CUDA is unavailable. Check the PyTorch installation before continuing."
    )

Libraries imported successfully.

PyTorch version: 2.11.0+cu128
CUDA available: True
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU
GPU memory: 6.0 GB


In [2]:
# ============================================================
# 2. LOAD EXISTING DATASET
# ============================================================

DATASET_PATH = Path(
    "pubmed_medical_affairs_3layer_corpus_FINAL.csv"
)

if not DATASET_PATH.exists():
    raise FileNotFoundError(
        f"Dataset not found at: {DATASET_PATH.resolve()}\n"
        "Update DATASET_PATH to the correct CSV location."
    )

df = pd.read_csv(DATASET_PATH)

required_columns = ["pmid", "title", "abstract", "layer"]

missing_columns = [
    col for col in required_columns
    if col not in df.columns
]

if missing_columns:
    raise ValueError(
        f"Required columns are missing: {missing_columns}"
    )

df["abstract"] = df["abstract"].fillna("").astype(str).str.strip()

print("Dataset loaded successfully.")
print("Dataset shape:", df.shape)
print("\nRecords per analytical layer:")
print(df["layer"].value_counts())

display(df[["pmid", "title", "abstract", "layer"]].head(3))

Dataset loaded successfully.
Dataset shape: (21256, 6)

Records per analytical layer:
layer
L1_scientific_engagement    9599
L3_comparative_critical     5849
L2_safety_risk              5808
Name: count, dtype: int64


,pmid,title,abstract,layer
0,41483736,Intracranial activity of antibody-drug conjuga...,Antibody-drug conjugates (ADCs) represent a ra...,L1_scientific_engagement
1,41483728,Effect of quetiapine versus haloperidol on del...,INTRODUCTION: Delirium is a neuropsychiatric c...,L1_scientific_engagement
2,41483727,Vision-language models in diagnostic imaging: ...,BACKGROUND: Radiology faces an unprecedented w...,L1_scientific_engagement


In [3]:
# ============================================================
# 3. INSTALLATION AND MODEL PREPARATION
# ============================================================

from sentence_transformers import SentenceTransformer

# Biomedical sentence-embedding model
MODEL_NAME = "pritamdeka/S-PubMedBert-MS-MARCO"

print("Selected model:", MODEL_NAME)

# Load model onto the available GPU
model = SentenceTransformer(
    MODEL_NAME,
    device="cuda"
)

print("\nModel loaded successfully.")
print("Model device:", model.device)
print("Embedding dimension:", model.get_sentence_embedding_dimension())

W1024 12:15:40.728000 6920 site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


Selected model: pritamdeka/S-PubMedBert-MS-MARCO


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]


Model loaded successfully.
Model device: cuda:0
Embedding dimension: 768


C:\Users\DELL\AppData\Local\Temp\ipykernel_6920\913803766.py:20: FutureWarning: The `get_sentence_embedding_dimension` method has been renamed to `get_embedding_dimension`.
  print("Embedding dimension:", model.get_sentence_embedding_dimension())


In [4]:
# ============================================================
# 4. GENERATE BIOMEDICAL ABSTRACT EMBEDDINGS
# ============================================================

import numpy as np
import pandas as pd
from pathlib import Path

# ------------------------------------------------------------
# 4.1 Prepare the abstracts
# ------------------------------------------------------------

embedding_df = df.copy()

# Preserve the original row number for traceability
embedding_df["source_row"] = np.arange(len(embedding_df))

# Exclude only records with genuinely empty abstracts
embedding_df = embedding_df[
    embedding_df["abstract"].notna()
    & embedding_df["abstract"].astype(str).str.strip().ne("")
].copy()

abstracts = embedding_df["abstract"].astype(str).tolist()

print("Abstracts to encode:", len(abstracts))
print("GPU:", torch.cuda.get_device_name(0))

# ------------------------------------------------------------
# 4.2 Generate embeddings
# ------------------------------------------------------------

embeddings = model.encode(
    abstracts,
    batch_size=16,
    show_progress_bar=False,
    convert_to_numpy=True,
    normalize_embeddings=True
)

print("\nEmbedding generation completed.")
print("Embedding matrix shape:", embeddings.shape)
print("Embedding data type:", embeddings.dtype)

# ------------------------------------------------------------
# 4.3 Validate the output
# ------------------------------------------------------------

assert embeddings.shape[0] == len(embedding_df), (
    "Mismatch between abstracts and embeddings."
)

assert np.isfinite(embeddings).all(), (
    "Embeddings contain non-finite values."
)

print("Validation passed: every retained abstract has an embedding.")

# ------------------------------------------------------------
# 4.4 Save embeddings and metadata
# ------------------------------------------------------------

OUTPUT_DIR = Path("biomedical_embedding_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

np.save(
    OUTPUT_DIR / "abstract_embeddings.npy",
    embeddings
)

metadata_columns = [
    "source_row",
    "pmid",
    "title",
    "year",
    "journal",
    "layer"
]

embedding_df[metadata_columns].to_csv(
    OUTPUT_DIR / "embedding_metadata.csv",
    index=False
)

print("\nSaved files:")
print(OUTPUT_DIR / "abstract_embeddings.npy")
print(OUTPUT_DIR / "embedding_metadata.csv")

Abstracts to encode: 21256
GPU: NVIDIA GeForce RTX 3050 6GB Laptop GPU

Embedding generation completed.
Embedding matrix shape: (21256, 768)
Embedding data type: float32
Validation passed: every retained abstract has an embedding.

Saved files:
biomedical_embedding_outputs\abstract_embeddings.npy
biomedical_embedding_outputs\embedding_metadata.csv


In [5]:
# ============================================================
# 5. SEMANTIC SIMILARITY ANALYSIS
# ============================================================

import numpy as np
import pandas as pd

from pathlib import Path
from IPython.display import display
from sklearn.preprocessing import normalize

# ------------------------------------------------------------
# 5.1 Load the saved embeddings and metadata
# ------------------------------------------------------------

OUTPUT_DIR = Path("biomedical_embedding_outputs")

embeddings = np.load(
    OUTPUT_DIR / "abstract_embeddings.npy",
    mmap_mode="r"
)

metadata = pd.read_csv(
    OUTPUT_DIR / "embedding_metadata.csv"
).reset_index(drop=True)

print("Embeddings loaded:", embeddings.shape)
print("Metadata loaded:", metadata.shape)

assert embeddings.shape[0] == len(metadata), (
    "The number of embeddings does not match the metadata."
)

required_columns = {"pmid", "title", "layer"}
assert required_columns.issubset(metadata.columns), (
    f"Metadata must contain these columns: {required_columns}"
)

assert metadata["layer"].nunique() == 3, (
    "Expected three analytical layers."
)

# ------------------------------------------------------------
# 5.2 Select approximately 10,000 abstracts proportionally
#     across the three layers
# ------------------------------------------------------------

RANDOM_STATE = 42
TARGET_SAMPLE_SIZE = 10_000

if len(metadata) < TARGET_SAMPLE_SIZE:
    raise ValueError(
        f"Only {len(metadata)} records are available."
    )

layer_counts = metadata["layer"].value_counts().sort_index()

# Allocate the sample proportionally to each layer's size.
exact_allocations = (
    layer_counts / len(metadata) * TARGET_SAMPLE_SIZE
)

sample_allocations = np.floor(
    exact_allocations
).astype(int)

# Distribute any remaining places using the largest remainders.
remaining = TARGET_SAMPLE_SIZE - sample_allocations.sum()

remainders = (
    exact_allocations - sample_allocations
).sort_values(ascending=False)

for layer in remainders.index[:remaining]:
    sample_allocations[layer] += 1

sampled_parts = []

for layer, n_to_sample in sample_allocations.items():
    layer_rows = metadata.index[
        metadata["layer"] == layer
    ]

    sampled_parts.append(
        metadata.loc[layer_rows].sample(
            n=int(n_to_sample),
            random_state=RANDOM_STATE
        )
    )

sample_metadata = (
    pd.concat(sampled_parts)
    .sample(frac=1, random_state=RANDOM_STATE)
    .reset_index(drop=True)
)

sample_indices = sample_metadata.index.to_numpy()

# The embedding rows correspond to the original metadata row order.
# Use the original row positions saved before resetting the index.
# Reconstruct these positions using PMID, which should be unique.
pmid_to_row = pd.Series(
    np.arange(len(metadata)),
    index=metadata["pmid"].astype(str)
)

sample_original_indices = (
    sample_metadata["pmid"]
    .astype(str)
    .map(pmid_to_row)
    .to_numpy()
)

if pd.isna(sample_original_indices).any():
    raise ValueError("Could not map all sampled PMIDs to embeddings.")

sample_embeddings = np.asarray(
    embeddings[sample_original_indices],
    dtype=np.float32
)

sample_metadata["sample_index"] = np.arange(
    len(sample_metadata)
)

print("\nTotal sampled abstracts:", len(sample_metadata))
print("\nSample allocation by layer:")
display(
    sample_metadata["layer"]
    .value_counts()
    .sort_index()
    .to_frame("abstract_count")
)

# ------------------------------------------------------------
# 5.3 Prepare embeddings and configure pair sampling
# ------------------------------------------------------------

# Normalise vectors so their dot product equals cosine similarity.
sample_embeddings = normalize(
    sample_embeddings,
    norm="l2",
    axis=1
).astype(np.float32)

# Target controlled pair samples, rather than storing every
# possible pair among the 10,000 abstracts.
TARGET_WITHIN_PAIRS = 100_000
TARGET_BETWEEN_PAIRS = 100_000
PAIR_RANDOM_STATE = 42
SIMILARITY_BATCH_SIZE = 10_000

rng = np.random.default_rng(PAIR_RANDOM_STATE)
n_sample = len(sample_metadata)

layer_indices = {
    layer: np.flatnonzero(
        sample_metadata["layer"].to_numpy() == layer
    )
    for layer in sample_metadata["layer"].unique()
}

# ------------------------------------------------------------
# 5.4 Sample unique pairs and calculate cosine similarity
# ------------------------------------------------------------

def sample_unique_pairs(
    indices_a,
    indices_b,
    n_pairs,
    rng,
    same_group=False
):
    """
    Randomly sample unique pairs of sample-row indices.
    Self-pairs and duplicate unordered pairs are excluded.
    """
    indices_a = np.asarray(indices_a)
    indices_b = np.asarray(indices_b)

    if same_group:
        possible_pairs = len(indices_a) * (len(indices_a) - 1) // 2
    else:
        possible_pairs = len(indices_a) * len(indices_b)

    n_pairs = min(int(n_pairs), int(possible_pairs))

    if n_pairs == 0:
        return np.empty((0, 2), dtype=np.int32)

    selected = set()

    while len(selected) < n_pairs:
        remaining = n_pairs - len(selected)
        batch_n = max(10_000, remaining * 2)

        a = rng.choice(indices_a, size=batch_n, replace=True)
        b = rng.choice(indices_b, size=batch_n, replace=True)

        if same_group:
            valid = a != b
            a, b = a[valid], b[valid]

        lo = np.minimum(a, b)
        hi = np.maximum(a, b)

        # Encode each unordered pair as a unique integer.
        keys = lo.astype(np.int64) * n_sample + hi.astype(np.int64)

        for key in np.unique(keys):
            selected.add(int(key))
            if len(selected) >= n_pairs:
                break

    keys = np.fromiter(
        selected,
        dtype=np.int64,
        count=n_pairs
    )

    return np.column_stack(
        (keys // n_sample, keys % n_sample)
    ).astype(np.int32)


layers = sorted(layer_indices.keys())

# Allocate within-layer pairs proportionally to the number
# of possible within-layer pairs.
within_capacity = {
    layer: len(layer_indices[layer]) *
           (len(layer_indices[layer]) - 1) // 2
    for layer in layers
}

total_within_capacity = sum(within_capacity.values())
within_allocations = {}

for layer in layers:
    within_allocations[layer] = int(
        TARGET_WITHIN_PAIRS *
        within_capacity[layer] /
        total_within_capacity
    )

# Distribute rounding remainder.
within_remainder = (
    TARGET_WITHIN_PAIRS - sum(within_allocations.values())
)

for layer in sorted(
    layers,
    key=lambda x: (
        TARGET_WITHIN_PAIRS *
        within_capacity[x] /
        total_within_capacity
        - within_allocations[x]
    ),
    reverse=True
)[:within_remainder]:
    within_allocations[layer] += 1

pair_arrays = []
pair_types = []

for layer in layers:
    pairs = sample_unique_pairs(
        layer_indices[layer],
        layer_indices[layer],
        within_allocations[layer],
        rng,
        same_group=True
    )
    pair_arrays.append(pairs)
    pair_types.extend(["Within-layer"] * len(pairs))

# Allocate between-layer pairs equally across the three
# possible layer combinations.
layer_combinations = [
    (layers[0], layers[1]),
    (layers[0], layers[2]),
    (layers[1], layers[2])
]

between_base = TARGET_BETWEEN_PAIRS // len(layer_combinations)
between_remainder = TARGET_BETWEEN_PAIRS % len(layer_combinations)

for combo_idx, (layer_a, layer_b) in enumerate(layer_combinations):
    n_for_combo = between_base + (
        1 if combo_idx < between_remainder else 0
    )

    pairs = sample_unique_pairs(
        layer_indices[layer_a],
        layer_indices[layer_b],
        n_for_combo,
        rng,
        same_group=False
    )

    pair_arrays.append(pairs)
    pair_types.extend(["Between-layer"] * len(pairs))

pair_indices = np.vstack(pair_arrays)
pairwise_results = pd.DataFrame({
    "index_1": pair_indices[:, 0],
    "index_2": pair_indices[:, 1],
    "pair_type": pair_types
})

# Record metadata for each member of each pair.
for suffix, index_column in [
    ("1", "index_1"),
    ("2", "index_2")
]:
    pair_rows = pairwise_results[index_column].to_numpy()
    pairwise_results[f"pmid_{suffix}"] = (
        sample_metadata.iloc[pair_rows]["pmid"].to_numpy()
    )
    pairwise_results[f"title_{suffix}"] = (
        sample_metadata.iloc[pair_rows]["title"].to_numpy()
    )
    pairwise_results[f"layer_{suffix}"] = (
        sample_metadata.iloc[pair_rows]["layer"].to_numpy()
    )

# Calculate dot products in batches to avoid creating a full
# 10,000 x 10,000 similarity matrix.
similarities = np.empty(len(pairwise_results), dtype=np.float32)

for start in range(0, len(pairwise_results), SIMILARITY_BATCH_SIZE):
    end = min(
        start + SIMILARITY_BATCH_SIZE,
        len(pairwise_results)
    )

    batch_pairs = pair_indices[start:end]
    left = sample_embeddings[batch_pairs[:, 0]]
    right = sample_embeddings[batch_pairs[:, 1]]

    similarities[start:end] = np.sum(
        left * right,
        axis=1
    )

pairwise_results["similarity"] = similarities

print("\nSampled abstracts:", len(sample_metadata))
print("Unique pairs analysed:", len(pairwise_results))
print("\nPairs by type:")
display(
    pairwise_results["pair_type"]
    .value_counts()
    .to_frame("pair_count")
)

print("\nBetween-layer pairs by layer combination:")
display(
    pairwise_results.loc[
        pairwise_results["pair_type"] == "Between-layer"
    ]
    .groupby(["layer_1", "layer_2"])
    .size()
    .to_frame("pair_count")
)

# ------------------------------------------------------------
# 5.5 Summarise similarity
# ------------------------------------------------------------

print("\nOverall pairwise similarity summary:")
display(
    pairwise_results["similarity"]
    .describe()
    .to_frame("cosine_similarity")
)

print("\nSimilarity by pair type:")
similarity_summary = (
    pairwise_results.groupby("pair_type")["similarity"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(4)
)

display(similarity_summary)

print("\nSimilarity by layer combination:")
layer_pair_summary = (
    pairwise_results.groupby(["layer_1", "layer_2"])["similarity"]
    .agg(["count", "mean", "median", "std", "min", "max"])
    .round(4)
)

display(layer_pair_summary)

# ------------------------------------------------------------
# 5.6 Save outputs
# ------------------------------------------------------------

pairwise_results.to_csv(
    OUTPUT_DIR / "semantic_similarity_pairs_sample.csv",
    index=False
)

similarity_summary.to_csv(
    OUTPUT_DIR / "semantic_similarity_summary.csv"
)

layer_pair_summary.to_csv(
    OUTPUT_DIR / "semantic_similarity_by_layer_pair.csv"
)

sample_metadata.to_csv(
    OUTPUT_DIR / "semantic_similarity_abstract_sample.csv",
    index=False
)

print("\nSaved outputs:")
print(OUTPUT_DIR / "semantic_similarity_pairs_sample.csv")
print(OUTPUT_DIR / "semantic_similarity_summary.csv")
print(OUTPUT_DIR / "semantic_similarity_by_layer_pair.csv")
print(OUTPUT_DIR / "semantic_similarity_abstract_sample.csv")


Embeddings loaded: (21256, 768)
Metadata loaded: (21256, 6)

Total sampled abstracts: 10000

Sample allocation by layer:


,abstract_count
layer,
L1_scientific_engagement,4516
L2_safety_risk,2732
L3_comparative_critical,2752



Sampled abstracts: 10000
Unique pairs analysed: 200000

Pairs by type:


,pair_count
pair_type,
Within-layer,100000
Between-layer,100000



Between-layer pairs by layer combination:


...


Similarity by pair type:


,count,mean,median,std,min,max
pair_type,,,,,,
Between-layer,100000,0.9052,0.9051,0.0141,0.8399,0.9782
Within-layer,100000,0.9077,0.9077,0.0144,0.8362,0.9878



Similarity by layer combination:


count    mean  median  \
layer_1                  layer_2                                           
L1_scientific_engagement L1_scientific_engagement  57563  0.9103  0.9102   
                         L2_safety_risk            16802  0.9077  0.9076   
                         L3_comparative_critical   16866  0.9050  0.9049   
L2_safety_risk           L1_scientific_engagement  16532  0.9075  0.9075   
                         L2_safety_risk            21064  0.9060  0.9058   
                         L3_comparative_critical   16844  0.9029  0.9029   
L3_comparative_critical  L1_scientific_engagement  16467  0.9049  0.9048   
                         L2_safety_risk            16489  0.9030  0.9029   
                         L3_comparative_critical   21373  0.9025  0.9024   

                                                      std     min     max  
layer_1                  layer_2                                           
L1_scientific_engagement L1_scientific_engagement  0.0138  0.8510  0.9878  
                         L2_safety_risk            0.0139  0.8566  0.9782  
                         L3_comparative_critical   0.0138  0.8405  0.9668  
L2_safety_risk           L1_scientific_engagement  0.0139  0.8503  0.9772  
                         L2_safety_risk            0.0143  0.8459  0.9782  
                         L3_comparative_critical   0.0141  0.8504  0.9670  
L3_comparative_critical  L1_scientific_engagement  0.0139  0.8399  0.9596  
                         L2_safety_risk            0.0140  0.8421  0.9663  
                         L3_comparative_critical   0.0145  0.8362  0.9741


Saved outputs:
biomedical_embedding_outputs\semantic_similarity_pairs_sample.csv
biomedical_embedding_outputs\semantic_similarity_summary.csv
biomedical_embedding_outputs\semantic_similarity_by_layer_pair.csv
biomedical_embedding_outputs\semantic_similarity_abstract_sample.csv


In [6]:
# ============================================================
# 6. INSPECT THE MOST AND LEAST SIMILAR ABSTRACT PAIRS
# ============================================================

from IPython.display import display

# ------------------------------------------------------------
# 6.1 Most similar pairs
# ------------------------------------------------------------

print("TOP 10 MOST SIMILAR ABSTRACT PAIRS")

top_pairs = (
    pairwise_results
    .sort_values("similarity", ascending=False)
    [[
        "pmid_1",
        "title_1",
        "layer_1",
        "pmid_2",
        "title_2",
        "layer_2",
        "similarity",
        "pair_type"
    ]]
    .head(10)
    .reset_index(drop=True)
)

display(top_pairs)

# ------------------------------------------------------------
# 6.2 Least similar pairs
# ------------------------------------------------------------

print("\nTOP 10 LEAST SIMILAR ABSTRACT PAIRS")

bottom_pairs = (
    pairwise_results
    .sort_values("similarity", ascending=True)
    [[
        "pmid_1",
        "title_1",
        "layer_1",
        "pmid_2",
        "title_2",
        "layer_2",
        "similarity",
        "pair_type"
    ]]
    .head(10)
    .reset_index(drop=True)
)

display(bottom_pairs)

# ------------------------------------------------------------
# 6.3 Save the inspection tables
# ------------------------------------------------------------

OUTPUT_DIR = Path("biomedical_embedding_outputs")

top_pairs.to_csv(
    OUTPUT_DIR / "top_10_similar_abstract_pairs.csv",
    index=False
)

bottom_pairs.to_csv(
    OUTPUT_DIR / "bottom_10_similar_abstract_pairs.csv",
    index=False
)

print("\nInspection tables saved successfully.")

TOP 10 MOST SIMILAR ABSTRACT PAIRS


,pmid_1,title_1,layer_1,pmid_2,title_2,layer_2,similarity,pair_type
0,41148842,Heat Shock Proteins in Pancreatic Cancer: Path...,L1_scientific_engagement,41309192,Heat shock proteins (HSPs) as chaperones for o...,L1_scientific_engagement,0.987764,Within-layer
1,41402630,[Side effects of immune checkpoint inhibitors ...,L2_safety_risk,41077682,Risk factors and mechanisms of immune checkpoi...,L2_safety_risk,0.978189,Within-layer
2,41226036,Advancing HBV Diagnostics: The Role of Ultrase...,L1_scientific_engagement,41472024,Advances in Novel Detection Technologies for O...,L2_safety_risk,0.978174,Between-layer
3,41340634,Predicability of PD-L1 expression in cancer ce...,L1_scientific_engagement,41115616,Predictive models for immune checkpoint inhibi...,L1_scientific_engagement,0.977853,Within-layer
4,41039945,Remimazolam: a comprehensive review.,L2_safety_risk,41029206,Efficacy and safety of remimazolam for inducti...,L2_safety_risk,0.977676,Within-layer
5,41369205,Screening and management of metabolic dysfunct...,L2_safety_risk,41151689,Pros and cons of different dietary patterns fo...,L1_scientific_engagement,0.977199,Between-layer
6,41020003,A systematic review and meta-analysis on the e...,L2_safety_risk,41272492,Comparative efficacy and safety of finerenone ...,L1_scientific_engagement,0.975031,Between-layer
7,41457847,[Colchicine in Cardiology Practice. Mechanisms...,L1_scientific_engagement,41190671,Inflammation and atherosclerotic cardiovascula...,L1_scientific_engagement,0.974792,Within-layer
8,41096874,Cardiovascular Effects of Cannabidiol: From Mo...,L2_safety_risk,41288593,Cannabidiol in Gliomas: Therapeutic Potential ...,L1_scientific_engagement,0.974370,Between-layer
9,41187481,CAR-T cell therapy: A therapeutic strategy for...,L1_scientific_engagement,41345813,"Hepatic Effects, Potential Drug-Induced Liver ...",L1_scientific_engagement,0.974260,Within-layer



TOP 10 LEAST SIMILAR ABSTRACT PAIRS


,pmid_1,title_1,layer_1,pmid_2,title_2,layer_2,similarity,pair_type
0,41416330,Neuroprotective Potential of Sodium-Glucose Co...,L3_comparative_critical,41436046,[The applications and challenges of rapid dete...,L3_comparative_critical,0.836220,Within-layer
1,41270707,Multiple realization in cognitive neurorehabil...,L3_comparative_critical,41369852,"Emergency laparoscopic ""hybrid"" intraperitonea...",L3_comparative_critical,0.837277,Within-layer
2,41266001,Prostate Cancer Grading: An Update.,L3_comparative_critical,41165325,The Story behind the Science: A history of Q-V...,L1_scientific_engagement,0.839881,Between-layer
3,41132036,Foreign Body Granulomas Reaction Related to Co...,L1_scientific_engagement,41436046,[The applications and challenges of rapid dete...,L3_comparative_critical,0.840519,Between-layer
4,41433258,Core training elicits greater improvements tha...,L3_comparative_critical,41043504,Airway exposure to microplastics: Potential me...,L2_safety_risk,0.842122,Between-layer
5,41343018,Recent progress in biowaste-derived carbon dot...,L2_safety_risk,41274657,Rescue at Extreme Altitude: Managing the Ethic...,L2_safety_risk,0.845885,Within-layer
6,41415768,Evolution of Gleason Score Prostate - A Review.,L3_comparative_critical,41313822,The blue nexus unveiled: Interlinking marine p...,L3_comparative_critical,0.847263,Within-layer
7,41397806,[Research progress on augmentation repair of a...,L3_comparative_critical,41357332,Epidemiology of childhood cancer in Asia.,L3_comparative_critical,0.848680,Within-layer
8,41207778,Climate Change and Dermatologic Health in Peop...,L3_comparative_critical,41370495,Comparison of vaginal natural orifice translum...,L3_comparative_critical,0.849587,Within-layer
9,41326059,Controlled release system of pesticide nanopar...,L2_safety_risk,41064936,International Contrast Ultrasound Society Safe...,L2_safety_risk,0.849973,Within-layer



Inspection tables saved successfully.
